In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Transactions Ingest")
    # Parquet + AQE + Broadcast = speed
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.parquet.filterPushdown", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.sql.shuffle.partitions", "200")   # tune to cluster size
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)


spark.conf.set("spark.sql.caseSensitive", "false")
spark

25/10/09 16:25:59 WARN Utils: Your hostname, I-Spider-Spacy-Y.local resolves to a loopback address: 127.0.0.1; using 10.240.65.140 instead (on interface en0)
25/10/09 16:25:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 16:25:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/09 16:25:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/10/09 16:25:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/10/09 16:25:59 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pathlib import Path
import pandas as pd

In [3]:
# input paths
P2_path = "../data/given/part 2/tables/transactions_20210228_20210827_snapshot/"
P3_path = "../data/given/part 3/tables/transactions_20210828_20220227_snapshot/"
P4_path = "../data/given/part 4/tables/transactions_20220228_20220828_snapshot/"\


# where to save the cleaned, combined dataset (partitioned by merchant_abn)
OUT_PATH = "../data/raw/transactions_cleaned_parquet/"



In [4]:
spark.read.parquet(P2_path)

user_id,merchant_abn,dollar_value,order_id,order_datetime
18478,62191208634,63.255848959735246,949a63c8-29f7-4ab...,2021-08-20
2,15549624934,130.3505283105634,6a84c3cf-612a-457...,2021-08-20
18479,64403598239,120.15860593212783,b10dcc33-e53f-425...,2021-08-20
3,60956456424,136.6785200286976,0f09c5a5-784e-447...,2021-08-20
18479,94493496784,72.96316578355305,f6c78c1a-4600-4c5...,2021-08-20
3,76819856970,448.529684285612,5ace6a24-cdf0-4aa...,2021-08-20
18479,67609108741,86.4040605836911,d0e180f0-cb06-42a...,2021-08-20
3,34096466752,301.5793450525113,6fb1ff48-24bb-4f9...,2021-08-20
18482,70501974849,68.75486276223054,8505fb33-b69a-412...,2021-08-20
4,49891706470,48.89796461900801,ed11e477-b09f-4ae...,2021-08-20


In [5]:
spark.read.parquet(P2_path).printSchema()
spark.read.parquet(P3_path).printSchema()
spark.read.parquet(P4_path).printSchema()

root
 |-- user_id: long (nullable = true)
 |-- merchant_abn: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)

root
 |-- user_id: long (nullable = true)
 |-- merchant_abn: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)

root
 |-- user_id: long (nullable = true)
 |-- merchant_abn: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



In [6]:

# --- 1) Load and harmonize schema ------------------------------------------
# (If schemas differ slightly, select columns in a common order.)
base_cols = ["user_id", "merchant_abn", "dollar_value", "order_id", "order_datetime"]

p2 = spark.read.parquet(P2_path).select(*base_cols).withColumn("source_batch", F.lit("P2"))
p3 = spark.read.parquet(P3_path).select(*base_cols).withColumn("source_batch", F.lit("P3"))
p4 = spark.read.parquet(P4_path).select(*base_cols).withColumn("source_batch", F.lit("P4"))

raw = p2.unionByName(p3).unionByName(p4)

In [7]:
# --- 2) Basic type enforcement / normalization -----------------------------
# Ensure order_datetime is TimestampType (or DateType if that’s what you want).
# Your preview shows DateType; here we promote to timestamp at midnight for flexibility.


raw = raw.withColumn(
    "order_date", F.to_timestamp(F.col("order_datetime").cast("string"))
).drop("order_datetime").withColumnRenamed("order_date", "order_datetime")


In [8]:

# --- 3) Sanity filters (tweak as needed) -----------------------------------
# Rules:
#  - non-null user_id keys
#  - positive amount (allow zero? set >= 0 if needed)
#  - date within the expected window
AMOUNT_MIN = 0.00
DATE_MIN = F.to_timestamp(F.lit("2021-02-28"))
DATE_MAX = F.to_timestamp(F.lit("2022-08-28"))  # inclusive upper bound by day

clean = (
    raw
    .filter(F.col("user_id").isNotNull() & (F.col("user_id") > 0))
    .filter(F.col("merchant_abn").isNotNull() & (F.col("merchant_abn") > 0))
    .filter(F.col("order_id").isNotNull() & (F.length(F.col("order_id")) > 0))
    .filter(F.col("dollar_value").isNotNull() & (F.col("dollar_value") >= AMOUNT_MIN))
    .filter(F.col("order_datetime").isNotNull())
    .filter( (F.col("order_datetime") >= DATE_MIN) & (F.col("order_datetime") <= DATE_MAX))
)

In [9]:
# Drop duplicate transactions.
# Assume order_id is globally unique:
dedup = clean.dropDuplicates(["order_id"])


In [10]:
# --- 4) Quick QA counts -----------------------------------------------------
def show_counts(label, df):
    print(f"[{label}] rows = {df.count():,}")
show_counts("RAW", raw)
show_counts("CLEAN", clean)
show_counts("DEDUP", dedup)

[RAW] rows = 14,195,505


[CLEAN] rows = 12,561,377


[DEDUP] rows = 12,561,377


In [11]:

# --- 5) Repartition and write by merchant ----------------------------------
# Choose a partition count that matches your cluster & data size
# (4x–6x number of cores is a decent starting point)
dedup = dedup.repartition("order_datetime")

# Save partitioned by merchant_abn; overwrite is typical for refreshable pipelines.
(dedup
 .write
 .mode("overwrite")
 .partitionBy("order_datetime")
 .parquet(OUT_PATH))